In [1]:
import json
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

import config
from input.input import load_raw_data
from model import generics, single_ml_model_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from utils.compare_fs_vs_baseline import build_comparison
from utils.export_metrics_to_csv import save_csv

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen impor

In [2]:
# === Notebook de FS na janela de 10% (pct10) -- MLP single (SKlearnModel) / rf_embedded ===
# Mesma estrutura dos chamados_v4_fs_* ('auto'), mas: (a) lag_size_override =
# resolve_lag_size_pct(N - test_size, 0.10) por serie; (b) comparacao par-a-par
# SEMPRE contra o baseline pct10 DA PROPRIA FAMILIA (celula final), NUNCA
# contra o baseline 'auto'. experiment_id/model_name distintos, sem underscore.
# seed=42 (familia MLP). force=False. Unica acao: Restart Kernel -> Run All.
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='rf_embedded')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt', 'windspeedfortaleza.txt', 'samurec.txt']

experiment_id = 'chamados_pct10_fs_mlp_rfembedded'
model_name = 'mlppct10rfembedded'   # -> 1mlppct10rfembedded.pkl (sem underscore, RUNBOOK.md 7)
normalize = True
force = False
model_exec = 10

experiment_params = {
    'diff_kpss': False,
    'horizon': 1,
    'type_filter': None,
}

model_parameters = {
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
}

# Comparacao par-a-par: baseline pct10 da PROPRIA familia (Parte 1). Nunca 'auto'.
baseline_experiment_id = 'chamados_pct10'
baseline_model_name = '1mlppct10'
linear_model_name_to_exclude = None

experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

In [3]:
# Sanity-check (mesmo padrao dos notebooks 'auto'): Pipeline.get_params(deep=True)
# expoe as chaves que GridSearch vai usar; e strategy <-> experiment_id/model_name
# consistentes entre si.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f'get_params(deep=True) nao expos: {missing}'

strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, f'{strategy_slug!r} nao em experiment_id={experiment_id!r}'
assert strategy_slug in model_name, f'{strategy_slug!r} nao em model_name={model_name!r}'
print(f'OK -- strategy={model.named_steps["selector"].strategy!r} consistente; keys expostas.')

OK -- strategy='rf_embedded' consistente; keys expostas.


In [4]:
# Janela de 10%: lag_size_override = resolve_lag_size_pct(N - test_size, 0.10)
# por serie -- MESMA base que get_max_lag_to_consider (PACF sobre
# ts_univariate[0:-test_size]). Computado aqui, nada a editar.
# force=False: execution() (correcao de 2026-09-02) pula .pkl ja existente
# e nao-vazio -- re-Run All e idempotente.
lag_pct_por_serie = {}
for base_name in series_list:
    n_raw = len(load_raw_data(base_name))
    n_train = n_raw - int(config.TEST_SIZE * n_raw)
    lag_pct = grid_search_exp.resolve_lag_size_pct(n_train, pct=0.10)
    lag_pct_por_serie[base_name] = lag_pct
    print(f'{base_name}  N={n_raw}  N-test={n_train}  lag_pct={lag_pct}')
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
        lag_size_override=lag_pct,
        estimator_random_state_base=grid_search_exp.MLP_RANDOM_STATE_BASE,  # CLAUDE.md 3.4 -- seed fixa das familias MLP
    )
    exec_gs.execution()

airlines.txt  N=144  N-test=130  lag_pct=13
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000}
austres.txt  N=89  N-test=81  lag_pct=8
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000}
coloradoRiver.txt  N=744  N-test=670  lag_pct=67
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000}
sunspot.txt  N=288  N-test=260  lag_pct=26
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000}
windspeedfortaleza.txt  N=144  N-test=130  lag_pct=13
{'estimator__hidden_layer_sizes': 10, 'estimator__max_iter': 1000}
samurec.txt  N=1188  N-test=1070  lag_pct=107
{'estimator__hidden_layer_sizes': 20, 'estimator__max_iter': 1000}


In [5]:
from utils.export_metrics_to_csv import run_export_metrics_to_csv

df_metrics = run_export_metrics_to_csv(
    experiment_dir, experiment_dir_results / 'metrics.csv', detail=True,
)
df_metrics

[INFO] 6 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_mlp_rfembedded'.

  OK  airlines_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  austres_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  coloradoRiver_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  samurec_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  sunspot_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  windspeedfortaleza_1mlppct10rfembedded.pkl  ->  10 linha(s)

[OK] CSV agregado (média das repetições) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_rfembedded\metrics.csv
     6 linha(s) × 20 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_rfembedded\metrics_detail.csv
     60 linha(s) × 13 coluna(s)


,ExperimentID,Serie,Modelo,N_Repeticoes,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,theil_mean,theil_std,ARV_mean,ARV_std,IA_mean,IA_std,POCID_mean,POCID_std
0,chamados_pct10_fs_mlp_rfembedded,airlines,1mlppct10rfembedded,10,446.760223,141.807058,20.941750,3.019078,18.079488,2.453348,3.948848,0.492443,0.221446,0.112100,0.083272,0.031510,0.979687,0.007233,77.857143,2.258770
1,chamados_pct10_fs_mlp_rfembedded,austres,1mlppct10rfembedded,10,1059.367336,863.210847,30.163553,12.889592,26.946931,12.381872,0.153759,0.070667,0.491027,0.408709,0.075935,0.058360,0.977022,0.018474,87.500000,0.000000
2,chamados_pct10_fs_mlp_rfembedded,coloradoRiver,1mlppct10rfembedded,10,0.039997,0.003587,0.199807,0.009086,0.165248,0.007886,19.002712,0.786765,1.439324,0.161873,0.653288,0.028478,0.783326,0.015218,59.324324,0.767089
3,chamados_pct10_fs_mlp_rfembedded,samurec,1mlppct10rfembedded,10,48.835538,0.883620,6.987987,0.062997,5.654712,0.050185,19.312024,0.177615,10.185568,1.397394,12.360169,1.035084,0.311441,0.017337,56.016949,2.093066
4,chamados_pct10_fs_mlp_rfembedded,sunspot,1mlppct10rfembedded,10,478.420858,15.455648,21.870268,0.353169,17.174082,0.314590,50.878585,2.656662,0.788952,0.032048,0.326296,0.016629,0.927204,0.003126,80.000000,3.450328
5,chamados_pct10_fs_mlp_rfembedded,windspeedfortaleza,1mlppct10rfembedded,10,0.189794,0.006974,0.435585,0.008173,0.395932,0.011630,14.422929,0.391966,1.215822,0.015090,0.465245,0.007118,0.893615,0.003192,50.714286,4.054616


In [6]:
from utils.export_selected_features import run_export_selected_features

df_features = run_export_selected_features(
    experiment_dir, experiment_dir_results / 'selected_features.csv', detail=True,
)
df_features

[INFO] 6 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_mlp_rfembedded'.

  OK  airlines_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  austres_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  coloradoRiver_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  samurec_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  sunspot_1mlppct10rfembedded.pkl  ->  10 linha(s)
  OK  windspeedfortaleza_1mlppct10rfembedded.pkl  ->  10 linha(s)

[OK] CSV agregado (média/desvio por série × modelo) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_rfembedded\selected_features.csv
     6 linha(s) × 8 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_rfembedded\selected_features_detail.csv
     60 linha(s) × 9 coluna(s)


,ExperimentID,Serie,Modelo,Strategy,N_Features_Selected_mean,N_Features_Selected_std,N_Repeticoes,N_Features_Total
0,chamados_pct10_fs_mlp_rfembedded,airlines,1mlppct10rfembedded,rf_embedded,1.3,0.483046,10,13
1,chamados_pct10_fs_mlp_rfembedded,austres,1mlppct10rfembedded,rf_embedded,4.0,0.666667,10,8
2,chamados_pct10_fs_mlp_rfembedded,coloradoRiver,1mlppct10rfembedded,rf_embedded,6.2,0.421637,10,67
3,chamados_pct10_fs_mlp_rfembedded,samurec,1mlppct10rfembedded,rf_embedded,39.9,2.766867,10,107
4,chamados_pct10_fs_mlp_rfembedded,sunspot,1mlppct10rfembedded,rf_embedded,3.0,0.000000,10,26
5,chamados_pct10_fs_mlp_rfembedded,windspeedfortaleza,1mlppct10rfembedded,rf_embedded,3.0,0.000000,10,13


In [7]:
# Comparacao PAR-A-PAR: FS pct10 x baseline pct10 da MESMA familia.
# NUNCA contra o baseline 'auto' -- isola o efeito da selecao de features
# do efeito da definicao de janela. A chave do dict e so o rotulo da coluna
# de saida (o slug da estrategia).
fs_dirs = {experiment_id.rsplit('_', 1)[-1]: experiment_dir}
df_cmp = build_comparison(
    Path(config.MODEL_DATA_PATH) / baseline_experiment_id,
    fs_dirs,
    baseline_model_name=baseline_model_name,
    linear_model_name_to_exclude=linear_model_name_to_exclude,
)
save_csv(df_cmp, experiment_dir_results / 'comparison.csv',
         label='comparacao FS pct10 x baseline pct10 (par-a-par)')
df_cmp

[INFO] 30 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10'.

  OK  airlines_1amv1pct10.pkl  ->  10 linha(s)
  OK  airlines_1arima.pkl  ->  1 linha(s)
  OK  airlines_1aspct10.pkl  ->  1 linha(s)
  OK  airlines_1mlppct10.pkl  ->  10 linha(s)
  OK  airlines_1svrpct10.pkl  ->  1 linha(s)
  OK  austres_1amv1pct10.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  1 linha(s)
  OK  austres_1aspct10.pkl  ->  1 linha(s)
  OK  austres_1mlppct10.pkl  ->  10 linha(s)
  OK  austres_1svrpct10.pkl  ->  1 linha(s)
  OK  coloradoRiver_1amv1pct10.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  1 linha(s)
  OK  coloradoRiver_1aspct10.pkl  ->  1 linha(s)
  OK  coloradoRiver_1mlppct10.pkl  ->  10 linha(s)
  OK  coloradoRiver_1svrpct10.pkl  ->  1 linha(s)
  OK  samurec_1amv1pct10.pkl  ->  10 linha(s)
  OK  samurec_1arima.pkl  ->  1 linha(s)
  OK  samurec_1aspct10.pkl  ->  1 linha(s)
  OK  samurec_1mlppct10.pkl  ->  10 linha(s)
  OK  samurec_1svr

,Serie,Baseline_RMSE,rfembedded_RMSE,rfembedded_PctGain,rfembedded_NFeatures
0,airlines,28.856466,20.941750,27.427879,1.3
1,austres,24.680016,30.163553,-22.218527,4.0
2,coloradoRiver,0.210362,0.199807,5.017336,6.2
3,samurec,7.187309,6.987987,2.773247,39.9
4,sunspot,19.222508,21.870268,-13.774266,3.0
5,windspeedfortaleza,0.367736,0.435585,-18.450279,3.0


In [8]:
import json

metadata = {
    'experiment_id': experiment_id,
    'notebook': 'single_models/mlp_pct10_rf_embedded.ipynb',
    'tipo': 'fs pct10',
    'familia': 'MLP single (SKlearnModel) / rf_embedded',
    'janela': 'pct10 -- resolve_lag_size_pct(N - int(config.TEST_SIZE*N), 0.10)',
    'lag_pct_por_serie': {s: lag_pct_por_serie[s] for s in series_list},
    'series': series_list,
    'model_exec': model_exec,
    'seed': grid_search_exp.MLP_RANDOM_STATE_BASE,
    'model_parameters': model_parameters,
    'diff_kpss': experiment_params['diff_kpss'],
    'baseline_pareado': baseline_model_name,
    'strategy': 'rf_embedded',
}
experiment_dir_results.mkdir(parents=True, exist_ok=True)
(experiment_dir_results / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, default=str), encoding='utf-8'
)
print('metadata.json ->', experiment_dir_results / 'metadata.json')

Failed to read module file 'C:\Projetos\mestrado_codigos\experiments\src\model\hybrid_system_exp.py' for module 'model.hybrid_system_exp': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 39146: character maps to <undefined>


metadata.json -> C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_rfembedded\metadata.json
